# Notebook 6: Machine Learning Pipeline & Model Training
**Project**: Loan Approval Prediction & Banking Analytics

---
## 1. Introduction
In this notebook, we build and train machine learning models. 

### Modeling Strategy:
1. **Pre-processing**: Scale numerical features and encode categorical features.
2. **Train-Test Split**: Stratified 80-20 split.
3. **Algorithms**:
   - Logistic Regression
   - Decision Tree Classifier
   - Random Forest Classifier
   - XGBoost Classifier
   - CatBoost Classifier
4. **Validation**: K-Fold Cross-Validation.
5. **Hyperparameter Tuning**: GridSearchCV / RandomizedSearchCV.
6. **Model Serialization**: Save models for evaluations.


In [1]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Model imports
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Load processed data
data_path = os.path.join("..", "Dataset", "processed", "loan_processed.csv")
df = pd.read_csv(data_path)
print(f"Processed data shape: {df.shape}")

Processed data shape: (614, 20)


## 2. Separate Features and Target

In [2]:
# Define features and target
X = df.drop(['Loan_ID', 'Loan_Status'], axis=1)
y = df['Loan_Status']

# List categorical and numerical features
categorical_features = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 
                        'Property_Area', 'IncomeCategory', 'ApplicantType', 'RiskCategory']
numerical_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 
                      'Credit_History', 'TotalIncome', 'Loan_to_Income_Ratio', 'EMI_Estimate', 'FamilySize']

## 3. Data Preprocessing Pipeline

In [3]:
# Define transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

# 80-20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")

Training shape: (491, 18)
Testing shape: (123, 18)


## 4. Inception of Baseline Models & Cross-Validation

In [4]:
# Define base models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0)
}

# 5-Fold CV evaluation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    clf = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    scores = cross_val_score(clf, X_train, y_train, cv=kf, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name} CV Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

Logistic Regression CV Accuracy: 0.7842 (+/- 0.0608)
Decision Tree CV Accuracy: 0.6945 (+/- 0.0490)


Random Forest CV Accuracy: 0.7536 (+/- 0.0539)


XGBoost CV Accuracy: 0.7516 (+/- 0.0532)


CatBoost CV Accuracy: 0.7740 (+/- 0.0591)


## 5. Hyperparameter Tuning

### Tuning Random Forest

In [5]:
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), 
                                ('model', RandomForestClassifier(random_state=42))])

param_grid_rf = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [5, 8, 12, None],
    'model__min_samples_split': [2, 5, 10]
}

grid_rf = GridSearchCV(rf_pipeline, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print("Best RF Parameters:", grid_rf.best_params_)
print("Best RF CV Accuracy:", grid_rf.best_score_)

Best RF Parameters: {'model__max_depth': 8, 'model__min_samples_split': 10, 'model__n_estimators': 200}
Best RF CV Accuracy: 0.7963512677798391


### Tuning XGBoost

In [6]:
xgb_pipeline = Pipeline(steps=[('preprocessor', preprocessor), 
                                 ('model', XGBClassifier(random_state=42, eval_metric='logloss'))])

param_grid_xgb = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.8, 1.0]
}

grid_xgb = GridSearchCV(xgb_pipeline, param_grid_xgb, cv=5, scoring='accuracy', n_jobs=-1)
grid_xgb.fit(X_train, y_train)

print("Best XGB Parameters:", grid_xgb.best_params_)
print("Best XGB CV Accuracy:", grid_xgb.best_score_)

Best XGB Parameters: {'model__learning_rate': 0.01, 'model__max_depth': 5, 'model__n_estimators': 100, 'model__subsample': 1.0}
Best XGB CV Accuracy: 0.7963306534735106


### Tuning CatBoost

In [7]:
cat_pipeline = Pipeline(steps=[('preprocessor', preprocessor), 
                                 ('model', CatBoostClassifier(random_state=42, verbose=0))])

param_grid_cat = {
    'model__iterations': [100, 200, 300],
    'model__depth': [4, 6, 8],
    'model__learning_rate': [0.01, 0.05, 0.1]
}

grid_cat = GridSearchCV(cat_pipeline, param_grid_cat, cv=5, scoring='accuracy', n_jobs=-1)
grid_cat.fit(X_train, y_train)

print("Best CatBoost Parameters:", grid_cat.best_params_)
print("Best CatBoost CV Accuracy:", grid_cat.best_score_)

Best CatBoost Parameters: {'model__depth': 4, 'model__iterations': 100, 'model__learning_rate': 0.01}
Best CatBoost CV Accuracy: 0.7983714698000413


## 6. Serializing Tuned Models

In [8]:
# Save models and test data splits for evaluation
os.makedirs("../Models", exist_ok=True)

# Save best pipelines
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', models['Logistic Regression'])])
lr_pipeline.fit(X_train, y_train)
with open('../Models/logistic_regression.pkl', 'wb') as f:
    pickle.dump(lr_pipeline, f)

dt_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', models['Decision Tree'])])
dt_pipeline.fit(X_train, y_train)
with open('../Models/decision_tree.pkl', 'wb') as f:
    pickle.dump(dt_pipeline, f)

with open('../Models/random_forest_tuned.pkl', 'wb') as f:
    pickle.dump(grid_rf.best_estimator_, f)

with open('../Models/xgboost_tuned.pkl', 'wb') as f:
    pickle.dump(grid_xgb.best_estimator_, f)

with open('../Models/catboost_tuned.pkl', 'wb') as f:
    pickle.dump(grid_cat.best_estimator_, f)

# Save test sets as well for the next notebook
test_data = {'X_test': X_test, 'y_test': y_test}
with open('../Models/test_splits.pkl', 'wb') as f:
    pickle.dump(test_data, f)

# Determine the best model and save it as best_model.pkl
best_model = grid_xgb.best_estimator_ if grid_xgb.best_score_ >= grid_cat.best_score_ else grid_cat.best_estimator_
with open('../Models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("All tuned models and testing subsets successfully serialized!")

All tuned models and testing subsets successfully serialized!
